In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
import os
from langchain_google_genai import GoogleGenerativeAI


In [9]:
load_dotenv()
google_api_key=os.getenv('GOOGLE_GEMINI_API_KEY')
gemini_model=GoogleGenerativeAI(model='gemini-1.5-pro', temperature=0.7, google_api_key=google_api_key)



In [ ]:
class EssayState(TypedDict):
    topic:str
    essay:str
    cot:str
    doa:str
    lang:str
    final_evaluation:str

In [ ]:
def cot(state:EssayState)->EssayState:
    topic=state['topic']
    essay=state['essay']

    prompt=f"given topic with essay, give the feedback on the essay in terms of clearity of thought, in the text(within 20 words) and numerical score(between 0 to 10)"

    res=gemini_model.invoke(prompt)

    return {'cot':res}

def doa(state:EssayState)->EssayState:
    topic=state['topic']
    essay=state['essay']

    prompt=f"given topic with essay, give the feedback on the essay in terms of depth of analysis, in the text(within 20 words) and numerical score(between 0 to 10)"

    res=gemini_model.invoke(prompt)

    return {'doa':res}

def lang(state:EssayState)->EssayState:
    topic=state['topic']
    essay=state['essay']

    prompt=f"given topic with essay, give the feedback on the essay in terms of language use, in the text(within 20 words) and numerical score(between 0 to 10)"

    res=gemini_model.invoke(prompt)

    return {'lang':res}

def final_evaluation(state:EssayState)->EssayState:
    topic=state['topic']
    essay=state['essay']
    cot=state['cot']
    doa=state['doa']
    lang=state['lang']

    prompt=f"given topic with essay, and feedback on clarity of thought, depth of analysis and language use, give the final evaluation of the essay in terms of overall score(between 0 to 10) and summary in 20 words"

    res=gemini_model.invoke(prompt)

    return {'final_evaluation':res}

In [ ]:
graph=StateGraph(EssayState)

# add node

graph.add_node('cot', cot)
graph.add_node('doa', doa)
graph.add_node('lang', lang)
graph.add_node('final_evaluation', final_evaluation)

# add edge
graph.add_edge(START, 'cot')
graph.add_edge(START, 'doa')
graph.add_edge(START, 'lang')

graph.add_edge('cot', 'final_evaluation')
graph.add_edge('doa', 'final_evaluation')
graph.add_edge('lang', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow=graph.compile()
